# 규칙 기반 추천 결과 평가

4장에서 만든 규칙 기반 추천 결과를 바탕으로, 추천일 이후 기록이 어떤 흐름으로 이어졌는지 사후적으로 확인한다.

이 장의 평가는 추천을 생성할 때 사용한 입력값이 아니다. 추천일 이후 데이터를 평가 목적으로만 사용한다.

In [5]:
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()

rule_based_output_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_rule_based_recommendations.csv"
)

rule_based_output_df = pd.read_csv(
    rule_based_output_path,
    parse_dates=["recommendation_date"],
)

rule_based_output_df.shape

(100, 15)

In [6]:
rides_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_bike_rides_cleaned.csv"
)

rides_df = pd.read_csv(
    rides_path,
    parse_dates=["date"],
)

daily_training_df = (
    rides_df
    .set_index("date")
    .resample("D")
    .agg(
        workout_hours=("workout_hours", "sum"),
        tss=("coggan_tss", "sum"),
        ride_count=("sport", "size"),
    )
    .sort_index()
)

daily_training_df.shape

(4407, 3)

In [7]:
daily_training_df["next_7_day_tss"] = (
    daily_training_df["tss"]
    .rolling(window=7, min_periods=7)
    .sum()
    .shift(-6)
)

daily_training_df[
    [
        "tss",
        "next_7_day_tss",
    ]
].loc["2009-03-21":"2009-03-25"].round(2)

,tss,next_7_day_tss
date,,
2009-03-21 00:00:00+00:00,194.46,435.28
2009-03-22 00:00:00+00:00,83.28,391.33
2009-03-23 00:00:00+00:00,0.00,458.54
2009-03-24 00:00:00+00:00,0.00,630.81
2009-03-25 00:00:00+00:00,0.00,630.81


In [8]:
future_tss_df = (
    daily_training_df[
        ["next_7_day_tss"]
    ]
    .reset_index()
    .rename(
        columns={
            "date": "recommendation_date",
        }
    )
)

evaluation_df = rule_based_output_df.merge(
    future_tss_df,
    on="recommendation_date",
    how="left",
)

evaluation_df[
    [
        "recommendation_date",
        "recommendation_context_type",
        "next_7_day_tss",
    ]
].head().round(
    {
        "next_7_day_tss": 2,
    }
)

,recommendation_date,recommendation_context_type,next_7_day_tss
0,2009-03-21 00:00:00+00:00,relatively_high_after_recorded,435.28
1,2009-03-22 00:00:00+00:00,relatively_high_after_recorded,391.33
2,2009-03-23 00:00:00+00:00,relatively_high_after_recorded,458.54
3,2009-03-24 00:00:00+00:00,relatively_high_after_no_record,630.81
4,2009-03-25 00:00:00+00:00,typical_range_after_no_record,630.81


In [11]:
print("추천 결과 평가용 표 크기", evaluation_df.shape)

evaluation_df[
    "next_7_day_tss"
].isna().sum()

추천 결과 평가용 표 크기 (100, 16)


np.int64(0)

In [12]:
future_tss_by_context_df = (
    evaluation_df
    .groupby("recommendation_context_type")[
        "next_7_day_tss"
    ]
    .agg(
        recommendation_days="size",
        mean_next_7_day_tss="mean",
        median_next_7_day_tss="median",
    )
    .round(2)
    .sort_values(
        "median_next_7_day_tss",
        ascending=False,
    )
)

future_tss_by_context_df

,recommendation_days,mean_next_7_day_tss,median_next_7_day_tss
recommendation_context_type,,,
relatively_low_after_no_record,9,632.81,724.92
typical_range_after_no_record,20,699.35,722.27
typical_range_after_recorded,38,721.09,712.11
relatively_high_after_recorded,19,658.08,705.14
relatively_low_after_recorded,10,628.05,660.69
relatively_high_after_no_record,4,711.42,647.87


In [13]:
future_tss_by_load_level_df = (
    evaluation_df
    .groupby("relative_recent_load_level")[
        "next_7_day_tss"
    ]
    .agg(
        recommendation_days="size",
        mean_next_7_day_tss="mean",
        median_next_7_day_tss="median",
    )
    .reindex(
        [
            "relatively_high",
            "typical_range",
            "relatively_low",
        ]
    )
    .round(2)
)

future_tss_by_load_level_df

,recommendation_days,mean_next_7_day_tss,median_next_7_day_tss
relative_recent_load_level,,,
relatively_high,23,667.36,675.71
typical_range,58,713.59,716.71
relatively_low,19,630.31,692.78


In [14]:
evaluation_df["next_7_day_tss_change"] = (
    evaluation_df["next_7_day_tss"]
    - evaluation_df["previous_7_day_tss"]
)

evaluation_df[
    [
        "recommendation_date",
        "previous_7_day_tss",
        "next_7_day_tss",
        "next_7_day_tss_change",
    ]
].head().round(
    {
        "previous_7_day_tss": 2,
        "next_7_day_tss": 2,
        "next_7_day_tss_change": 2,
    }
)

,recommendation_date,previous_7_day_tss,next_7_day_tss,next_7_day_tss_change
0,2009-03-21 00:00:00+00:00,829.25,435.28,-393.97
1,2009-03-22 00:00:00+00:00,926.31,391.33,-534.98
2,2009-03-23 00:00:00+00:00,812.04,458.54,-353.50
3,2009-03-24 00:00:00+00:00,709.89,630.81,-79.08
4,2009-03-25 00:00:00+00:00,580.91,630.81,49.90


In [15]:
tss_change_by_load_level_df = (
    evaluation_df
    .groupby("relative_recent_load_level")[
        "next_7_day_tss_change"
    ]
    .agg(
        recommendation_days="size",
        mean_tss_change="mean",
        median_tss_change="median",
    )
    .reindex(
        [
            "relatively_high",
            "typical_range",
            "relatively_low",
        ]
    )
    .round(2)
)

tss_change_by_load_level_df

,recommendation_days,mean_tss_change,median_tss_change
relative_recent_load_level,,,
relatively_high,23,-187.37,-154.76
typical_range,58,10.84,10.89
relatively_low,19,99.24,155.56


In [17]:
evaluation_df[
    "previous_7_day_tss"
].eq(0).sum()

evaluation_df["next_to_previous_7_day_tss_ratio"] = (
    evaluation_df["next_7_day_tss"]
    / evaluation_df["previous_7_day_tss"]
)

tss_ratio_by_load_level_df = (
    evaluation_df
    .groupby("relative_recent_load_level")[
        "next_to_previous_7_day_tss_ratio"
    ]
    .agg(
        recommendation_days="size",
        mean_tss_ratio="mean",
        median_tss_ratio="median",
    )
    .reindex(
        [
            "relatively_high",
            "typical_range",
            "relatively_low",
        ]
    )
    .round(2)
)

tss_ratio_by_load_level_df

,recommendation_days,mean_tss_ratio,median_tss_ratio
relative_recent_load_level,,,
relatively_high,23,0.79,0.83
typical_range,58,1.05,1.02
relatively_low,19,1.21,1.23


In [18]:
evaluation_output_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_recommendation_evaluation.csv"
)

evaluation_df.to_csv(
    evaluation_output_path,
    index=False,
)

evaluation_df.shape

(100, 18)

### 추천 결과 사후 평가 결과

- 추천일 당일과 이후 6일의 기록을 합산한 `next_7_day_tss`를 평가 지표로 만들고, 100개 추천일 모두에 연결했다. 이 값은 추천을 만들 때 사용한 입력값이 아니라 사후 평가용 결과값이다.
- 이전 7일 대비 이후 7일 TSS 변화량의 평균·중앙값은 높은 부하에서 각각 -187.37, -154.76, 일반 범위에서 10.84, 10.89, 낮은 부하에서 99.24, 155.56이었다.
- 이후 7일 TSS 비율의 평균·중앙값은 높은 부하에서 0.79, 0.83, 일반 범위에서 1.05, 1.02, 낮은 부하에서 1.21, 1.23이었다. 최근 부하가 높을수록 이후 기록 부하가 줄고, 낮을수록 늘어나는 관찰 패턴이 나타났다.
- 이 패턴은 규칙 기반 추천이 훈련 변화를 만들었다는 인과 증거가 아니다. 이전 부하에 따른 평균 회귀와 실제 훈련 계획 등 다른 요인이 함께 반영될 수 있다.
- 날짜별 추천 맥락과 사후 평가 지표를 포함한 100행 18열 결과를 `data/processed/goldencheetah_recommendation_evaluation.csv`로 저장했다.